# Chapter 3: Inside CNNs - Pooling & Parameters

In the previous chapter, we built our first CNN and saw that it works. But to build *deep* and *efficient* networks, we need to understand two critical concepts:
1.  **Parameters**: How heavy is our model? Why are CNNs so much lighter than MLPs?
2.  **Pooling**: How do we make the model robust to position changes (Translation Invariance)?

**Objective:**
-   Master the formula for calculating weights in `Dense` vs `Conv2D`.
-   Learn the difference between **Max Pooling** and **Average Pooling**.
-   Understand the standard "Funnel" Architecture of Deep CNNs.
-   Implement a deeper CNN with multiple `Conv2D` + `MaxPooling2D` blocks.

---

## 1. The Exploding Parameter Problem

Let's revisit why we moved away from Dense networks (MLPs) for images.
If we have a small 28x28 image, it has 784 pixels. 
If we connect this to a Dense layer with 10 neurons:
$$ Parameters = (Input \times Output) + Bias $$
$$ (784 \times 10) + 10 = 7850 $$

Not too bad. But what if we have a modern 4K image ($3840 \times 2160 \approx 8.3$ million pixels) and a layer of 1000 neurons?
$$ 8,300,000 \times 1000 \approx 8.3 \text{ Billion Parameters!} $$
This is unmanageable. 

**Enter Conv2D Effiency:**
A Convolutional layer **shares weights**. It doesn't care how big the image is. It only cares about the size of the *Kernel* (Filter).

### 1.1 The Formula
$$ Param_{Conv2D} = (K_h \times K_w \times C_{in} + 1) \times C_{out} $$

Where:
- $K_h, K_w$: Kernel Height/Width (e.g., 3x3)
- $C_{in}$: Input Channels (e.g., 3 for RGB)
- $C_{out}$: Output Channels (Number of Filters)
- $+1$: The Bias term per filter.

*Note: Notice that Image Height/Width are NOT in the formula!*

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, Input

# Let's verify this math with code

# Scenario: 10 Filters of size 3x3, Input is Grayscale (1 channel)
model_proof = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(filters=10, kernel_size=(3, 3))
])

# Manual Calculation:
# (3 * 3 * 1 + 1) * 10 = 10 * 10 = 100 parameters
model_proof.summary()

## 2. Pooling Layers: Downsampling

As we go deeper into a network, we want to summarize information. We don't need pixel-perfect precision; we need to know "Is there an ear here?". 

**Pooling** reduces the image size (usually by half) by keeping only the most important information.

### 2.1 Max Pooling
This is the most common type. It looks at a window (e.g., 2x2) and picks the **largest** number. 

![Max Pooling Diagram](https://upload.wikimedia.org/wikipedia/commons/e/e9/Max_pooling.png)
*Figure: Concept of Max Pooling - Reducing a 4x4 region to 2x2 by picking the max values.*
[Image Source](https://commons.wikimedia.org/)

### 2.2 Manual Max Pooling Implementation
Let's write a python function to perform Max Pooling (2x2) manually, just like we did with Convolution.

In [ ]:
import numpy as np

def manual_max_pooling(image_map, pool_size=2, stride=2):
    # Get dims
    h, w = image_map.shape
    
    # Output size: Half the input (if size=2, stride=2)
    new_h = h // pool_size
    new_w = w // pool_size
    
    output = np.zeros((new_h, new_w))
    
    for y in range(new_h):
        for x in range(new_w):
            # Start coordinates for this window
            start_y = y * stride
            start_x = x * stride
            
            # Extract 2x2 patch
            patch = image_map[start_y : start_y+pool_size, 
                              start_x : start_x+pool_size]
            
            # Take MAX
            output[y, x] = np.max(patch)
            
    return output

# Test it on a simple matrix
test_matrix = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 2],
    [8, 2, 1, 0],
    [1, 9, 3, 5]
])

print("Original (4x4):\n", test_matrix)
pooled = manual_max_pooling(test_matrix)
print("\nMax Pooled (2x2):\n", pooled)

# Check logic:
# Top-Left: max(1,3,5,6) -> 6
# Top-Right: max(2,4,1,2) -> 4
# Bot-Left: max(8,2,1,9) -> 9
# Bot-Right: max(1,0,3,5) -> 5

---

## 3. The "Funnel" Architecture

Standard CNNs follow a funnel shape:

![CNN Funnel Diagram](https://upload.wikimedia.org/wikipedia/commons/6/63/Typical_cnn.png)
*Figure: Deeper layers have smaller spatial Dimensions (W, H) but more Depth (Filters).*
[Image Source](https://commons.wikimedia.org/)

- **Width/Height**: Decrease (due to Pooling)
- **Depth (Filters)**: Increase (to capture more complex features like shapes -> objects -> faces)

Let's build a Deep CNN for Fashion MNIST that follows this pattern.

In [ ]:
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical

# 1. Load Data
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()
train_images = train_images.reshape((-1, 28, 28, 1)) / 255.0
test_images = test_images.reshape((-1, 28, 28, 1)) / 255.0
train_labels_oh = to_categorical(train_labels)
test_labels_oh = to_categorical(test_labels)

# 2. Build Deep Funnel Model
model = Sequential()

# Block 1
model.add(Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)))
model.add(MaxPooling2D((2, 2)))
# Shape is now (14, 14, 32)

# Block 2
# Note: We increase filters to 64
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D((2, 2)))
# Shape is now (7, 7, 64)

# Classifier
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(10, activation='softmax'))

model.summary()

### Observation on Summary
Look at the `Output Shape` column above:
1.  Starts at `(28, 28, 32)`
2.  Becomes `(14, 14, 32)` after first pool.
3.  Becomes `(7, 7, 64)` after second pool.

This confirms our "Funnel" design.

In [ ]:
# 3. Train the Deep Model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(train_images, train_labels_oh, epochs=5, validation_split=0.1)

## 4. Average Pooling vs Max Pooling

Max Pooling keeps the sharpest feature (max value). Average Pooling smoothes things out.

| Type | Action | Use Case |
| :--- | :--- | :--- |
| **Max Pooling** | Takes the largest pixel value. | Most common. Good for edges and distinct features. |
| **Avg Pooling** | Takes the average of pixels. | Used less often. Good for smoothing or global context. |

![Max vs Average Pooling](https://miro.medium.com/v2/resize:fit:1318/1*ypIfJX7iWX6h6Kbkfq85Kg.png)

*Figure: Comparison of Max vs Average Pooling.*
[Image Source](https://github.com/)


## 5. Exercise: Calculate Parameters

**Question:**
If you have a layer `Conv2D(Filters=20, kernel_size=(5,5))` receiving a 3-channel input ($RGB$), how many trainable parameters does it have?

Try to calculate it using the formula: $(K \times K \times C_{in} + 1) \times C_{out}$

In [ ]:
# Solution
# K = 5, Cin = 3, Cout = 20
parameters = (5 * 5 * 3 + 1) * 20
print(f" Calculated Parameters: {parameters}")

# Verification code
test_model = Sequential([
    Input(shape=(100, 100, 3)),
    Conv2D(20, (5, 5))
])
test_model.summary()